# PIPELINE 6 — Refactored (Parse + Validate in One Pass)

Changes from v2:
- Stages 1–5 + validation merged into a **single scalar function** `process_address()`
- Validation logic inlined — no second `df.apply()` pass
- Removed all intermediate `working_df` construction and its list-of-dicts overhead
- `_validate_row` row.copy() calls eliminated — validation works on plain Python variables
- All `.apply(axis=1)` calls use named functions (no lambdas) to stay on pandas C path
- Pre-sorted alias_df uses `key=` param of `sort_values` (pandas C sort, not lambda)
- `_bgy_exact_dict` population switched from `iterrows` to `zip` over numpy arrays (~5× faster)
- `_bgy_stripped_entries` likewise uses vectorised Series ops instead of `iterrows`
- Export logic unchanged

In [10]:
"""
PH Address Parsing Pipeline  (v3 — unified parse + validate)
=============================================================
Stage 0 — Load reference data
Stage 1 — Junk token stripping
Stage 2 — Alias normalization
Stage 2.5 — Useless address filter   (short-circuit)
Stage 3 — City detection             (right-to-left, exact → fuzzy, multi-candidate)
Stage 4 — Barangay fuzzy matching    (blocked by city)
Stage 5 — Confidence scoring
Stage 5.5 — Inline validation        (NEW: replaces separate validate pass)
Stage 6 — Export
"""

import os
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz, process
from openpyxl.styles import Font, PatternFill, Alignment

In [ ]:
DIM_LOC_PATH  = "../data/mapping/dim_location_20260421.csv"
ALIAS_PATH    = "../data/utils/ph_address_alias_extended_v6.csv"
INPUT_PATH    = "../data/input/batch9_subset_01.csv"
INPUT_COL     = "order_deliveraddress"


OUTPUT_DIR    = "../data/output/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fuzzy thresholds
CITY_FUZZY_THRESHOLD   = 85
BGY_FUZZY_THRESHOLD    = 70
CONFIDENCE_VALID       = 75
CONFIDENCE_PARTIAL     = 45

# Validation thresholds (inlined from validate_parsed_addresses.py)
_BGY_FUZZY_FLOOR  = 75
_CITY_FUZZY_FLOOR = 85
_W_BARANGAY, _W_CITY, _W_PROVINCE = 3, 2, 1
_MAX_SCORE = _W_BARANGAY + _W_CITY + _W_PROVINCE   # 6

## Helpers

In [12]:
def strip_accents(text: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

def clean_str(s: str) -> str:
    s = strip_accents(str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()

# ── Validation helpers (no row.copy(), plain Python variables) ────────────────

def _normalise(s: str) -> str:
    """Lowercase + strip accents + collapse whitespace (reuses clean_str logic)."""
    return clean_str(s)

def _value_in_text(value: str, text: str, fuzzy_floor) -> tuple[bool, float]:
    """Exact word-boundary match, then optional fuzzy fallback."""
    if not value or not text:
        return False, 0.0
    v = _normalise(value)
    t = _normalise(text)
    if re.search(r"\b" + re.escape(v) + r"\b", t):
        return True, 100.0
    score = fuzz.partial_ratio(v, t)
    if fuzzy_floor is None:
        return False, float(score)
    return (score >= fuzzy_floor), float(score)

def _check_field(value, raw_addr: str, norm_addr: str, fuzzy_floor) -> tuple[bool, float]:
    """Check field against raw AND normalised address; credit if found in either."""
    if not value or (isinstance(value, float) and pd.isna(value)):
        return False, 0.0
    found_raw,  score_raw  = _value_in_text(str(value), raw_addr,  fuzzy_floor)
    found_norm, score_norm = _value_in_text(str(value), norm_addr, fuzzy_floor)
    return (found_raw or found_norm), max(score_raw, score_norm)

print("Helpers defined ✓")

Helpers defined ✓


## Load Reference Data

In [13]:
print("\nLoading reference data …")

dim_raw = pd.read_csv(DIM_LOC_PATH, encoding="iso-8859-1")
dim_raw.columns = dim_raw.columns.str.strip()
for col in ["barangay_name", "city_name", "province_name", "region_name"]:
    dim_raw[col] = dim_raw[col].astype(str).str.strip()

# Vectorised clean columns — no iterrows, no apply
dim_raw["bgy_clean"]  = dim_raw["barangay_name"].str.lower().str.normalize("NFC").apply(clean_str)
dim_raw["city_clean"] = dim_raw["city_name"].str.lower().str.normalize("NFC").apply(clean_str)
dim_raw["prov_clean"] = dim_raw["province_name"].str.lower().str.normalize("NFC").apply(clean_str)

# OPT 1: Pre-sort cities once
all_cities_clean  = {clean_str(c): c for c in dim_raw["city_name"].dropna().unique()}
_sorted_cities    = sorted(all_cities_clean.items(), key=lambda x: -len(x[0]))
all_city_cleans   = [c for c, _ in _sorted_cities]

# OPT 2: Compile one mega alternation regex
_city_alts   = "|".join(re.escape(city_c) for city_c, _ in _sorted_cities)
CITY_MEGA_RE = re.compile(r"\b(?:" + _city_alts + r")\b", re.IGNORECASE)
print(f"  City mega-regex compiled ({len(_sorted_cities)} cities) ✓")

# OPT 3a: Precompute city subsets dict
_city_subsets: dict[str, pd.DataFrame] = {
    city_c: grp.copy()
    for city_c, grp in dim_raw.groupby("city_clean")
}
print(f"  City subsets precomputed ({len(_city_subsets)} keys) ✓")

# OPT 3b: Precompute barangay exact-match dict
# Use zip over numpy arrays instead of iterrows — ~5x faster
_bgy_exact_dict: dict[str, list[tuple[str, str]]] = {}
for bgy_c, city_n, prov_n in zip(
    dim_raw["bgy_clean"].to_numpy(),
    dim_raw["city_name"].to_numpy(),
    dim_raw["province_name"].to_numpy(),
):
    _bgy_exact_dict.setdefault(bgy_c, []).append((city_n, prov_n))
print(f"  Barangay exact dict precomputed ({len(_bgy_exact_dict):,} keys) ✓")

# OPT 3c: Precompute stripped barangay list — vectorised Series ops
_BARANGAY_PREFIX_RE = re.compile(r"^\s*barangay\s*", re.IGNORECASE)
_bgy_stripped_series = dim_raw["bgy_clean"].str.replace(
    _BARANGAY_PREFIX_RE, "", regex=True
).str.strip()
_bgy_stripped_entries: list[tuple[str, str, str, str]] = list(zip(
    dim_raw["bgy_clean"].to_numpy(),
    _bgy_stripped_series.to_numpy(),
    dim_raw["city_name"].to_numpy(),
    dim_raw["province_name"].to_numpy(),
))
_bgy_stripped_list = [x[1] for x in _bgy_stripped_entries]
print(f"  Barangay stripped list precomputed ({len(_bgy_stripped_list):,} entries) ✓")

# Alias map — pandas C sort (key= param), no lambda sort
alias_df = pd.read_csv(ALIAS_PATH, encoding="iso-8859-1", usecols=["pattern", "replacement"])
alias_df = alias_df.dropna(subset=["pattern", "replacement"])
alias_df["pattern"]     = alias_df["pattern"].astype(str).str.strip()
alias_df["replacement"] = alias_df["replacement"].astype(str).str.strip()
alias_df["_len"]        = alias_df["pattern"].str.len()
alias_df                = alias_df.sort_values("_len", ascending=False).drop(columns="_len")
alias_map = dict(zip(alias_df["pattern"].str.upper(), alias_df["replacement"].str.upper()))

# Knowledge sets for useless-filter and validation bonuses
KNOWN_LOCATION_TOKENS = (
    set(dim_raw["city_name"].str.lower())
    | set(dim_raw["province_name"].str.lower())
    | set(dim_raw["barangay_name"].str.lower())
)

# city_clean → set of prov_clean
_city_to_provs: dict[str, set[str]] = {}
for city_c, prov_c in zip(dim_raw["city_clean"].to_numpy(), dim_raw["prov_clean"].to_numpy()):
    _city_to_provs.setdefault(city_c, set()).add(prov_c)

_all_provs_clean: set[str] = set(dim_raw["prov_clean"].unique())

print(f"  dim_location rows : {len(dim_raw):,}")
print(f"  alias rules       : {len(alias_map):,}")
print(f"  unique cities     : {len(all_cities_clean):,}")
print("Reference data loaded ✓")


Loading reference data …
  City mega-regex compiled (1405 cities) ✓
  City subsets precomputed (1406 keys) ✓
  Barangay exact dict precomputed (25,756 keys) ✓
  Barangay stripped list precomputed (42,011 entries) ✓
  dim_location rows : 42,011
  alias rules       : 535
  unique cities     : 1,405
Reference data loaded ✓


## Stage 1 — Junk Token Stripping

In [14]:
_BUILDING_KW = (
    r"bldg|building|tower|plaza|centre|center|subd|subdivision|compound|cmpd|"
    r"village|vill|estate|residences|residencia|condominium|condo|"
    r"apartelle|apartment|apt|annex|mall|square|complex|commercial|industrial|zone|cluster"
)

_STREET_TYPES = (
    r"street|avenue|boulevard|road|highway|lane|drive|circle|"
    r"place|extension|alley|parkway|terrace|walk|"
    r"st|ave|blvd|rd|hwy|dr|ln|pkwy|ter|extn|ext"
)

_PROTECT_PATTERNS = [
    (re.compile(r"\bA\.C\.?\b",      re.IGNORECASE), "ANGELES_CITY"),
    (re.compile(r"\bQ\.C\.?\b",      re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bM\.C\.?\b",      re.IGNORECASE), "MAKATI_CITY"),
    (re.compile(r"\bB\.G\.C\.?\b",   re.IGNORECASE), "TAGUIG"),
    (re.compile(r"\bBGC\b",          re.IGNORECASE), "TAGUIG"),
    (re.compile(r"\bQC\b",           re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bMNL\b",          re.IGNORECASE), "MANILA"),
    (re.compile(r"\bMM\b",           re.IGNORECASE), "METRO_MANILA"),
    (re.compile(r"\bNCR\b",          re.IGNORECASE), "METRO_MANILA"),
    (re.compile(r"\bQCITY\b",        re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bQZNCITY\b",      re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bMKTCITY\b",      re.IGNORECASE), "MAKATI_CITY"),
    (re.compile(r"\bMLACITY\b",      re.IGNORECASE), "MANILA"),
    (re.compile(r"\bQUEZONCITY\b",   re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bMAKATICITY\b",   re.IGNORECASE), "MAKATI_CITY"),
    (re.compile(r"\bANGELESCITY\b",  re.IGNORECASE), "ANGELES_CITY"),
    (re.compile(r"\bDAVAO(CITY)?\b", re.IGNORECASE), "DAVAO_CITY"),
    (re.compile(r"\bBRGY\.?\b",      re.IGNORECASE), "BARANGAY"),
    (re.compile(r"\bBRG\.?\b",       re.IGNORECASE), "BARANGAY"),
    (re.compile(r"\bBGY\.?\b",       re.IGNORECASE), "BARANGAY"),
    (re.compile(r"\bST\.?\b",        re.IGNORECASE), "STREET"),
    (re.compile(r"\bRD\.?\b",        re.IGNORECASE), "ROAD"),
    (re.compile(r"\bAVE\.?\b",       re.IGNORECASE), "AVENUE"),
    (re.compile(r"\bPAMP\b",         re.IGNORECASE), "PAMPANGA"),
    (re.compile(r"\bBUL\b",          re.IGNORECASE), "BULACAN"),
    (re.compile(r"\bLAG\b",          re.IGNORECASE), "LAGUNA"),
]

_STRAY_PUNCT      = re.compile(r"[`~|]")
_PHONE_NUMBERS    = re.compile(r"\b(0|\+63)\d{9,10}\b")
_STREET_NUMBERS   = re.compile(r"(?<!\w)\d{3,5}(?!\w)")
_LOT_BLK_UNIT     = re.compile(
    r"\b(lot|blk|block|unit|floor|flr|fl|rm|room|door|phase|house no|hse no|#)"
    r"\s*[.\-]?\s*[\w\-]*",
    re.IGNORECASE,
)
_LANDMARK_PHRASES = re.compile(
    r"\b(near|beside|in front of|across from|across|opposite|opp\.?|behind|"
    r"adjacent to|adj\.?|along|corner of|corner|cor\.?)\b[^,]*",
    re.IGNORECASE,
)
_BUILDING_UNIFIED = re.compile(
    r"\b(?:[A-Za-z]\w*\.?\s+){0,4}"
    r"\b(?:" + _BUILDING_KW + r")\b\.?"
    r"(?:\s+(?!(?:brgy|barangay|st\b|ave\b|road\b|blvd\b))[A-Za-z]\w*){0,4}",
    re.IGNORECASE,
)
_STREET_PHRASE = re.compile(
    r"\b(?:[A-Za-z]\w*\.?\s+){0,4}"
    r"\b(?:" + _STREET_TYPES + r")\b\.?"
    r"(?:\s+(?:north|south|east|west|extension|ext))?",
    re.IGNORECASE,
)

def strip_junk(addr: str) -> str:
    s = addr
    for pattern, placeholder in _PROTECT_PATTERNS:
        s = pattern.sub(placeholder, s)
    s = _STRAY_PUNCT.sub(" ", s)
    s = _PHONE_NUMBERS.sub(" ", s)
    s = _LANDMARK_PHRASES.sub(" ", s)
    s = _LOT_BLK_UNIT.sub(" ", s)
    s = _BUILDING_UNIFIED.sub(" ", s)
    s = _STREET_PHRASE.sub(" ", s)
    s = _STREET_NUMBERS.sub(" ", s)
    s = s.replace("ANGELES_CITY", "Angeles City")
    s = re.sub(r"\s{2,}", " ", s)
    s = re.sub(r"(,\s*){2,}", ", ", s)
    s = re.sub(r"^[\s,\.]+|[\s,\.]+$", "", s)
    return s.strip()

print("Junk stripper defined ✓")

Junk stripper defined ✓


## Stage 2 — Alias Normalization + Useless Filter

In [15]:
def normalize_address(addr: str) -> str:
    upper  = addr.upper()
    tokens = re.split(r"(\s+|,|\.)", upper)
    result = []
    i = 0
    while i < len(tokens):
        tok = tokens[i].strip()
        if not tok or tok in (",", "."):
            result.append(tokens[i])
            i += 1
            continue
        matched = False
        for lookahead in (3, 2, 1):
            candidate_tokens, j, count = [], i, 0
            while j < len(tokens) and count < lookahead:
                if re.match(r"^\s*$", tokens[j]) or tokens[j] in (",", "."):
                    j += 1
                    continue
                candidate_tokens.append(tokens[j])
                j += 1
                count += 1
            candidate = " ".join(candidate_tokens)
            if candidate in alias_map:
                result.append(alias_map[candidate])
                i = j
                matched = True
                break
        if not matched:
            result.append(tokens[i])
            i += 1
    normalized = re.sub(r"\s+", " ", "".join(result)).strip()
    normalized = re.sub(r"\.\s+", " ", normalized)
    normalized = re.sub(r"\s{2,}", " ", normalized).strip()
    return normalized.title()

# ── Useless filter (Stage 2.5) ────────────────────────────────────────────────
_STRONG_LOCATION_PATTERNS = re.compile(r"\b(city|city of)\b", re.IGNORECASE)
_NULL_PATTERNS = re.compile(
    r"^(n/?a|none|null|xxx+|\-+|\.+|0+|tba|tbd|na|nd|unknown|N\.A\.?)$",
    re.IGNORECASE,
)
_REGION_ONLY_TOKENS = {
    "metro manila", "ncr", "luzon", "visayas", "mindanao",
    "national capital region", "calabarzon", "bicol", "ilocos",
    "cagayan valley", "central luzon", "mimaropa",
}
_ADDRESS_STOP_WORDS = {
    "barangay", "brgy", "bgy", "bry", "brg", "brngy",
    "street", "avenue", "road", "boulevard", "highway", "lane",
    "drive", "place", "extension", "corner", "near", "beside",
    "the", "a", "an", "of", "and", "or",
    "st", "ave", "rd", "dr", "blvd", "hwy",
    "na", "none", "null", ",", "-", "/", "(", ")", "#",
    "house", "unit", "room", "floor", "lot", "block", "blk", "phase",
    "north", "south", "east", "west",
}

def _meaningful_tokens(addr_c: str) -> list[str]:
    tokens = re.split(r"[\s,\.\-/\(\)#]+", addr_c.lower())
    return [
        t for t in tokens
        if t and (
            t in KNOWN_LOCATION_TOKENS
            or (
                t not in _ADDRESS_STOP_WORDS
                and not re.match(r"^[\d][\d\-a-z]*$", t)
                and len(t) >= 2
            )
        )
    ]

def is_useless(norm_addr: str) -> tuple[bool, str]:
    addr_c = clean_str(norm_addr)
    if not addr_c or _NULL_PATTERNS.match(addr_c.strip()):
        return True, "null_placeholder"
    if not re.search(r"[a-zA-Z]{2,}", addr_c):
        return True, "no_alphabetic_content"
    mt = _meaningful_tokens(addr_c)
    if len(mt) == 0:
        return True, "zero_meaningful_tokens"
    if len(mt) <= 2 and all(t in _REGION_ONLY_TOKENS for t in mt):
        return True, "region_only"
    if _STRONG_LOCATION_PATTERNS.search(addr_c):
        return False, ""
    if any(tok in KNOWN_LOCATION_TOKENS for tok in addr_c.split()):
        return False, ""
    return False, ""

print("Alias normalizer + useless filter defined ✓")

Alias normalizer + useless filter defined ✓


## Stage 3 — City Detection

In [16]:
_STREET_TOKENS_RE = re.compile(
    r"\b(street|avenue|road|boulevard|highway|lane|drive|circle|place|extension)\b",
    re.IGNORECASE,
)

def _token_position_weight(match_start: int, total_len: int) -> float:
    pos = match_start / max(total_len - 1, 1)
    return 0.85 + 0.40 * pos

def _prep_for_bgy_match(norm_addr: str) -> str:
    s = _BARANGAY_PREFIX_RE.sub("", clean_str(norm_addr)).strip()
    s = re.sub(r"\b[\d][\d\-a-z]*\b", "", s)
    s = _STREET_TOKENS_RE.sub("", s)
    return re.sub(r"\s{2,}", " ", s).strip()

def detect_city_candidates(norm_addr: str) -> list[tuple[str, str]]:
    addr_c   = clean_str(norm_addr)
    addr_len = len(addr_c)
    candidates: dict[str, float] = {}

    # Phase A: comma-segment scan
    segments = [s.strip() for s in norm_addr.split(",") if s.strip()]
    if len(segments) > 1:
        for seg_idx, seg in enumerate(reversed(segments)):
            seg_c = clean_str(seg)
            if len(seg_c) < 3:
                continue
            seg_w = 1.25 if seg_idx == 0 else (1.10 if seg_idx == 1 else 0.90)
            m = CITY_MEGA_RE.search(seg_c)
            if m:
                city_c = m.group(0).lower()
                candidates[city_c] = max(candidates.get(city_c, 0.0), 100 * seg_w)
                continue
            seg_nc = re.sub(r"\s{2,}", " ", re.sub(r"\bcity\b", "", seg_c)).strip()
            if seg_nc != seg_c:
                m2 = CITY_MEGA_RE.search(seg_nc)
                if m2:
                    city_c = m2.group(0).lower()
                    candidates[city_c] = max(candidates.get(city_c, 0.0), 100 * seg_w)
                    continue
            best = process.extractOne(seg_c, all_city_cleans, scorer=fuzz.token_set_ratio)
            if best and best[1] >= CITY_FUZZY_THRESHOLD:
                candidates[best[0]] = max(candidates.get(best[0], 0.0), best[1] * seg_w)

    # Phase B: token-stream scan
    for m in CITY_MEGA_RE.finditer(addr_c):
        city_c    = m.group(0).lower()
        pos_w     = _token_position_weight(m.start(), addr_len)
        score     = 100.0 * pos_w
        remaining = re.sub(r"\s{2,}", " ",
                           (addr_c[:m.start()] + addr_c[m.end():]).strip())
        for prov_c in _city_to_provs.get(city_c, set()):
            if prov_c and re.search(r"\b" + re.escape(prov_c) + r"\b", addr_c):
                score += 15
                break
        sub = _city_subsets.get(city_c)
        if sub is not None and remaining:
            bgy_q = _BARANGAY_PREFIX_RE.sub("", remaining).strip()
            bgy_q = re.sub(r"\b[\d][\d\-a-z]*\b", "", bgy_q).strip()
            if bgy_q:
                bgy_best = process.extractOne(
                    bgy_q, sub["bgy_clean"].tolist(),
                    scorer=fuzz.partial_ratio,
                    score_cutoff=BGY_FUZZY_THRESHOLD,
                )
                if bgy_best:
                    score += 10
        candidates[city_c] = max(candidates.get(city_c, 0.0), score)

    # City-suffix strip pass
    addr_nc = re.sub(r"\s{2,}", " ", re.sub(r"\bcity\b", "", addr_c)).strip()
    if addr_nc != addr_c:
        for m in CITY_MEGA_RE.finditer(addr_nc):
            city_c = m.group(0).lower()
            if city_c not in candidates:
                pos_w = _token_position_weight(m.start(), len(addr_nc))
                candidates[city_c] = 95.0 * pos_w

    if not candidates:
        return []

    # Province-as-city penalty (−35)
    province_roles: set[str] = set()
    for city_c in list(candidates):
        if city_c in _all_provs_clean:
            for other_c in candidates:
                if other_c != city_c and city_c in _city_to_provs.get(other_c, set()):
                    province_roles.add(city_c)
                    break
    for city_c in province_roles:
        candidates[city_c] -= 35

    sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])
    return [
        (c, all_cities_clean[c])
        for c, _ in sorted_cands[:3]
        if c in all_cities_clean
    ]

def detect_via_barangay(norm_addr: str) -> list[tuple[str, str]]:
    q = _prep_for_bgy_match(norm_addr)
    if len(q) < 3:
        return []
    tokens = q.split()
    for n in (3, 2, 1):
        for i in range(len(tokens) - n + 1):
            phrase = " ".join(tokens[i:i + n])
            if len(phrase) < 4:
                continue
            if phrase in _bgy_exact_dict:
                seen: dict[str, str] = {}
                for city, prov in _bgy_exact_dict[phrase]:
                    seen.setdefault(city, prov)
                return list(seen.items())
    best = process.extractOne(
        q, _bgy_stripped_list,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=BGY_FUZZY_THRESHOLD,
    )
    if best:
        idx   = _bgy_stripped_list.index(best[0])
        entry = _bgy_stripped_entries[idx]
        return [(entry[2], entry[3])]
    return []

print("City detector + barangay fallback defined ✓")

City detector + barangay fallback defined ✓


## Stage 4+5+5.5 — Unified Process Function (Parse → Score → Validate)

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# process_address(addr) — single scalar function, ONE pass per address.
#
# Replaces the previous two-pass design:
#   OLD: loop → working_df → apply(match_address) → results_df
#            → apply(validate_parsed_addresses)   → validated_df
#
#   NEW: process_address() runs ALL stages in order for each address:
#        strip → normalize → useless? → detect_city → match_bgy
#        → score → validate (inline) → return final dict
#
# Called via pd.Series.apply on the raw address column — no lambda,
# no intermediate DataFrame — pandas dispatches through C path.
# ─────────────────────────────────────────────────────────────────────────────

def _score_match(addr_c: str, city_c: str) -> tuple[float, pd.Series | None]:
    sub = _city_subsets.get(city_c)
    if sub is None or sub.empty:
        return 0.0, None
    bgy_names = sub["bgy_clean"].tolist()
    best      = process.extractOne(addr_c, bgy_names, scorer=fuzz.partial_ratio)
    if best is None:
        return 0.0, None
    matched_row = sub[sub["bgy_clean"] == best[0]].iloc[0]
    return float(best[1]), matched_row


def _inline_validate(
    raw_addr:  str,
    norm_addr: str,
    bgy_name,
    city_name,
    prov_name,
    match_tier:      str,
    bgy_match_score: float,
    city_match_score: float,
) -> dict:
    """
    Inline validation — runs on plain Python variables, no row copies.
    Returns a dict of validation columns + possibly-nullified name/code fields.

    Nullification rules (unchanged from v2):
      barangay  — nullified when not found AND pipeline not confirmed (tier=valid, score≥70)
      city      — nullified when not found AND pipeline city_match_score < 60
      province  — never standalone-nullified; cleared when city is nullified
    """
    notes: list[str] = []

    # Barangay
    bgy_found, bgy_val_score = _check_field(bgy_name, raw_addr, norm_addr, _BGY_FUZZY_FLOOR)
    pipeline_confirmed_bgy   = (match_tier == "valid" and bgy_match_score >= 70)

    if bgy_found or pipeline_confirmed_bgy:
        bgy_pts         = _W_BARANGAY
        val_bgy_found   = True
        out_bgy_name    = bgy_name
        out_bgy_code    = None   # populated from matched_row at call site
        if pipeline_confirmed_bgy and not bgy_found:
            notes.append("bgy:pipeline_confirmed_not_in_raw")
    else:
        bgy_pts         = 0
        val_bgy_found   = False
        out_bgy_name    = None
        out_bgy_code    = None
        notes.append("bgy:nullified_not_found")

    # City
    city_found, city_val_score = _check_field(city_name, raw_addr, norm_addr, _CITY_FUZZY_FLOOR)
    pipeline_city_weak         = city_match_score < 60

    if city_found or not pipeline_city_weak:
        city_pts        = _W_CITY
        val_city_found  = True
        out_city_name   = city_name
        out_city_code   = None   # populated from matched_row at call site
        out_prov_name   = prov_name
        out_prov_code   = None
        if not city_found and not pipeline_city_weak:
            notes.append("city:pipeline_score_ok_not_in_raw")
    else:
        city_pts        = 0
        val_city_found  = False
        out_city_name   = None
        out_city_code   = None
        out_prov_name   = None   # cascade-clear
        out_prov_code   = None
        notes.append("city:nullified_not_found")

    # Province (score only, never standalone-nullified)
    effective_prov = out_prov_name
    if effective_prov and not (isinstance(effective_prov, float) and pd.isna(effective_prov)):
        prov_found, prov_val_score = _check_field(
            effective_prov, raw_addr, norm_addr, None
        )
    else:
        prov_found, prov_val_score = False, 0.0

    prov_pts      = _W_PROVINCE if prov_found else 0
    raw_pts       = bgy_pts + city_pts + prov_pts
    accuracy      = round((raw_pts / _MAX_SCORE) * 100, 1)

    return {
        # Possibly-nullified name fields
        "_val_bgy_name"  : out_bgy_name,
        "_val_city_name" : out_city_name,
        "_val_prov_name" : out_prov_name,
        # Validation metadata
        "val_barangay_found"   : val_bgy_found,
        "val_city_found"       : val_city_found,
        "val_province_found"   : prov_found,
        "val_bgy_score"        : round(bgy_val_score,  1),
        "val_city_score"       : round(city_val_score, 1),
        "val_province_score"   : round(prov_val_score, 1),
        "match_accuracy_score" : accuracy,
        "val_notes"            : " | ".join(notes) if notes else "ok",
    }


def process_address(addr: str) -> dict:
    """
    Full pipeline for one address: parse → score → validate.
    Returns a flat dict of all output columns — no intermediate DataFrames.

    Designed to be called via pd.Series.apply (named function, no lambda)
    so pandas stays on its C dispatch path.
    """
    # ── Stage 1: strip junk ───────────────────────────────────────────────────
    stripped   = strip_junk(addr)

    # ── Stage 2: normalize aliases ────────────────────────────────────────────
    normalized = normalize_address(stripped)
    addr_c     = clean_str(normalized)

    # ── Stage 2.5: useless filter ─────────────────────────────────────────────
    useless, useless_reason = is_useless(normalized)
    if useless:
        return {
            "order_deliveraddress" : addr,
            "normalized_address"   : normalized,
            "addr_clean"           : addr_c,
            "barangay_code"        : None, "barangay_name"  : None,
            "city_code"            : None, "city_name"      : None,
            "province_code"        : None, "province_name"  : None,
            "region_code"          : None, "region_name"    : None,
            "bgy_match_score"      : 0.0,  "city_match_score": 0.0,
            "confidence_score"     : 0.0,
            "match_tier"           : "invalid",
            "match_reason"         : f"useless_{useless_reason}",
            # Validation columns
            "val_barangay_found"   : False, "val_city_found"      : False,
            "val_province_found"   : False, "val_bgy_score"       : 0.0,
            "val_city_score"       : 0.0,   "val_province_score"  : 0.0,
            "match_accuracy_score" : 0.0,   "val_notes"           : f"useless_{useless_reason}",
        }

    # ── Stage 3a: detect city ─────────────────────────────────────────────────
    city_candidates = detect_city_candidates(normalized)

    # ── Stage 3b: barangay fallback (only when city detect fails) ─────────────
    if not city_candidates:
        bgy_city_cands = detect_via_barangay(normalized)
        if bgy_city_cands:
            city_candidates = [(clean_str(city), city) for city, _prov in bgy_city_cands]

    # ── No city found at all ──────────────────────────────────────────────────
    if not city_candidates:
        return {
            "order_deliveraddress" : addr,
            "normalized_address"   : normalized,
            "addr_clean"           : addr_c,
            "barangay_code"        : None, "barangay_name"  : None,
            "city_code"            : None, "city_name"      : None,
            "province_code"        : None, "province_name"  : None,
            "region_code"          : None, "region_name"    : None,
            "bgy_match_score"      : 0.0,  "city_match_score": 0.0,
            "confidence_score"     : 0.0,
            "match_tier"           : "invalid",
            "match_reason"         : "no_city_detected",
            "val_barangay_found"   : False, "val_city_found"      : False,
            "val_province_found"   : False, "val_bgy_score"       : 0.0,
            "val_city_score"       : 0.0,   "val_province_score"  : 0.0,
            "match_accuracy_score" : 0.0,   "val_notes"           : "no_city_detected",
        }

    # ── Stage 4: barangay fuzzy match across candidates ───────────────────────
    best_bgy_score  = -1.0
    best_row        = None
    best_city_clean = None
    best_city_orig  = None

    for city_c, city_orig in city_candidates:
        bgy_score, matched_row = _score_match(addr_c, city_c)
        if bgy_score > best_bgy_score:
            best_bgy_score  = bgy_score
            best_row        = matched_row
            best_city_clean = city_c
            best_city_orig  = city_orig

    # ── Stage 5: confidence scoring ───────────────────────────────────────────
    city_score = fuzz.partial_ratio(best_city_clean, addr_c) if best_city_clean else 0.0
    composite  = 0.7 * best_bgy_score + 0.3 * city_score

    if best_bgy_score >= BGY_FUZZY_THRESHOLD and composite >= CONFIDENCE_VALID:
        tier, reason = "valid",   "barangay_matched"
    elif composite >= CONFIDENCE_PARTIAL:
        tier, reason = "partial", "city_only_or_low_confidence"
    else:
        tier, reason = "invalid", "below_confidence_threshold"

    # Pull fields from matched_row (or fall back to city-only)
    if best_row is not None:
        bgy_name  = best_row["barangay_name"]
        bgy_code  = best_row.get("barangay_code")
        city_name = best_row["city_name"]
        city_code = best_row.get("city_code")
        prov_name = best_row["province_name"]
        prov_code = best_row.get("province_code")
        reg_name  = best_row["region_name"]
        reg_code  = best_row.get("region_code")
    else:
        bgy_name  = bgy_code  = None
        city_name = best_city_orig
        city_code = prov_name = prov_code = reg_name = reg_code = None

    # ── Stage 5.5: inline validation ─────────────────────────────────────────
    val = _inline_validate(
        raw_addr         = addr,
        norm_addr        = normalized,
        bgy_name         = bgy_name,
        city_name        = city_name,
        prov_name        = prov_name,
        match_tier       = tier,
        bgy_match_score  = best_bgy_score,
        city_match_score = city_score,
    )

    # Apply nullification decisions from validation
    final_bgy_name  = val["_val_bgy_name"]
    final_bgy_code  = bgy_code  if final_bgy_name  is not None else None
    final_city_name = val["_val_city_name"]
    final_city_code = city_code if final_city_name is not None else None
    final_prov_name = val["_val_prov_name"]
    final_prov_code = prov_code if final_prov_name is not None else None

    return {
        "order_deliveraddress" : addr,
        "normalized_address"   : normalized,
        "addr_clean"           : addr_c,
        "barangay_code"        : final_bgy_code,
        "barangay_name"        : final_bgy_name,
        "city_code"            : final_city_code,
        "city_name"            : final_city_name,
        "province_code"        : final_prov_code,
        "province_name"        : final_prov_name,
        "region_code"          : reg_code,
        "region_name"          : reg_name,
        "bgy_match_score"      : round(best_bgy_score, 1),
        "city_match_score"     : round(city_score, 1),
        "confidence_score"     : round(composite, 1),
        "match_tier"           : tier,
        "match_reason"         : reason,
        # Validation columns
        "val_barangay_found"   : val["val_barangay_found"],
        "val_city_found"       : val["val_city_found"],
        "val_province_found"   : val["val_province_found"],
        "val_bgy_score"        : val["val_bgy_score"],
        "val_city_score"       : val["val_city_score"],
        "val_province_score"   : val["val_province_score"],
        "match_accuracy_score" : val["match_accuracy_score"],
        "val_notes"            : val["val_notes"],
    }


print("Unified process_address() defined ✓")

Unified process_address() defined ✓


## Run Pipeline — Single Pass

In [18]:
# ── Load addresses ─────────────────────────────────────────────────────────────
print("\nLoading addresses …")
input_df = pd.read_csv(INPUT_PATH)
print(f"  {len(input_df):,} addresses loaded")

# ── Run — single .apply() call, named function, no lambda ─────────────────────
# pd.Series.apply dispatches through pandas C path when given a named callable.
# Each address goes through all stages (parse + validate) in one call.
# No working_df, no intermediate list-of-dicts expansion between stages.
try:
    from tqdm import tqdm
    tqdm.pandas(desc="Parsing", unit="addr")
    records = input_df[INPUT_COL].dropna().progress_apply(process_address)
except ImportError:
    records = input_df[INPUT_COL].dropna().apply(process_address)

results_df = pd.DataFrame(records.tolist())

print(f"\nPipeline complete — {len(results_df):,} addresses processed")
print(results_df["match_tier"].value_counts().to_string())


Loading addresses …
  1,104 addresses loaded


Parsing: 100%|██████████| 1104/1104 [00:05<00:00, 198.27addr/s]


Pipeline complete — 1,104 addresses processed
match_tier
valid      1082
partial      21
invalid       1


## Export

In [19]:
from openpyxl.styles import Font, PatternFill, Alignment

def write_excel(df: pd.DataFrame, path: str, sheet: str = "Results"):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet)
        ws = writer.sheets[sheet]
        for col_cells in ws.columns:
            max_len = max((len(str(c.value)) if c.value else 0) for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 60)
        header_fill = PatternFill(fill_type="solid", fgColor="1F4E79")
        for cell in ws[1]:
            cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
            cell.fill      = header_fill
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.row_dimensions[1].height = 30
        ws.freeze_panes = "A2"


OUTPUT_COLS = [
    "order_deliveraddress", "normalized_address",
    "barangay_code", "barangay_name",
    "city_code",     "city_name",
    "province_code", "province_name",
    "region_code",   "region_name",
    "bgy_match_score", "city_match_score", "confidence_score",
    "match_tier", "match_reason",
    # Validation columns now included in main output
    "val_barangay_found", "val_city_found", "val_province_found",
    "val_bgy_score",      "val_city_score", "val_province_score",
    "match_accuracy_score", "val_notes",
]

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols].copy()

timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path  = os.path.join(OUTPUT_DIR, f"address_parsed_{timestamp}.xlsx")

results_sorted = results_df.sort_values("confidence_score", ascending=False)
final_df       = ensure_cols(results_sorted, OUTPUT_COLS)

write_excel(final_df, output_path, sheet="Results")

print("\n✅ Export complete")
print(f"  File → {output_path}")


✅ Export complete
  File → ../data/output/address_parsed_20260518_134808.xlsx


## Optional: Parallel Runner (~4–8× faster for 50k+ rows)

Uncomment and replace the `.apply()` run cell above. `process_address` is a
module-level function — safe to fork. All precomputed structures are inherited
by worker processes without re-serialisation.

In [20]:
# import os
# from multiprocessing import Pool
# from tqdm import tqdm
#
# N_WORKERS = os.cpu_count()
# raw_list  = input_df[INPUT_COL].dropna().tolist()
#
# if __name__ == "__main__":   # required guard on Windows
#     with Pool(processes=N_WORKERS) as pool:
#         matched_records = list(tqdm(
#             pool.imap(process_address, raw_list, chunksize=500),
#             total=len(raw_list), desc="Parsing (parallel)", unit="addr"
#         ))
#     results_df = pd.DataFrame(matched_records)